In [0]:
# Databricks notebook source

from __future__ import annotations

import uuid

# ============================================================
# Widgets
# ============================================================
dbutils.widgets.text("catalog", "phc_txdot")

catalog = dbutils.widgets.get("catalog").strip()

# ============================================================
# Constants
# ============================================================
SCRIPT_NAME = "250_build_silver_dim_item_specification.py"
RUN_ID = str(uuid.uuid4())

BASE_TABLE = f"{catalog}.silver.closed_bids_base_clean"
VENDOR_LATEST_TABLE = f"{catalog}.silver.closed_bids_vendor_item_latest"
ESTIMATE_CURRENT_TABLE = f"{catalog}.silver.closed_bids_estimate_item_current"
TARGET_TABLE = f"{catalog}.silver.dim_item_specification"
RUN_LOG_TABLE = f"{catalog}.staging.pipeline_run_log"

# ============================================================
# Helpers
# ============================================================
def sql_escape(value: str) -> str:
    return value.replace("'", "''")


def log_run(status: str, row_count: int | None, message: str) -> None:
    row_count_sql = "NULL" if row_count is None else str(row_count)
    spark.sql(f"""
        INSERT INTO {RUN_LOG_TABLE}
        VALUES (
              '{RUN_ID}'
            , '{SCRIPT_NAME}'
            , '{sql_escape(status)}'
            , {row_count_sql}
            , '{sql_escape(message)}'
            , current_timestamp()
        )
    """)

# ============================================================
# Build dim_item_specification
# ============================================================
try:
    spark.sql(f"""
    CREATE OR REPLACE TABLE {TARGET_TABLE}
    USING DELTA
    AS
    WITH spec_source AS (
        SELECT
              specification_code
            , specification_description
            , measurement_unit
            , spec_book_year
            , bid_code
            , bid_item_description
            , row_type
            , is_engineer_estimate_row
            , project_id
            , vendor_name_standardized
            , source_updated_at
            , ingested_at_utc
            , base_row_key
            , source_row_id
        FROM {BASE_TABLE}
        WHERE specification_code IS NOT NULL
          AND measurement_unit IS NOT NULL
    )

    , normalized AS (
        SELECT
              specification_code
            , specification_description
            , measurement_unit
            , spec_book_year
            , bid_code
            , bid_item_description
            , row_type
            , is_engineer_estimate_row
            , project_id
            , vendor_name_standardized
            , source_updated_at
            , ingested_at_utc
            , base_row_key
            , source_row_id

            , CASE
                  WHEN specification_description IS NOT NULL
                    THEN specification_description
                  WHEN COALESCE(bid_code, '') LIKE '9602-%'
                    THEN bid_item_description
                  WHEN COALESCE(bid_code, '') LIKE '9606-%'
                    THEN bid_item_description
                  WHEN UPPER(COALESCE(bid_item_description, '')) LIKE '%PAYMENT ADJUSTMENT%'
                    THEN bid_item_description
                  WHEN UPPER(COALESCE(bid_item_description, '')) LIKE '%SPECIAL DEDUCTION%'
                    THEN bid_item_description
                  ELSE NULL
              END AS effective_specification_description
        FROM spec_source
    )

    , keyed AS (
        SELECT
              specification_code
            , specification_description
            , effective_specification_description
            , measurement_unit
            , spec_book_year
            , bid_code
            , bid_item_description
            , row_type
            , is_engineer_estimate_row
            , project_id
            , vendor_name_standardized
            , source_updated_at
            , ingested_at_utc
            , base_row_key
            , source_row_id

            , TRIM(UPPER(effective_specification_description)) AS effective_specification_description_standardized
            , TRIM(UPPER(measurement_unit)) AS measurement_unit_standardized

            , md5(
                concat_ws(
                      '|'
                    , COALESCE(CAST(specification_code AS STRING), '')
                    , COALESCE(TRIM(UPPER(effective_specification_description)), '')
                    , COALESCE(TRIM(UPPER(measurement_unit)), '')
                )
              ) AS item_specification_key
        FROM normalized
        WHERE effective_specification_description IS NOT NULL
    )

    , canonical_bid_code_ranked AS (
        SELECT
              item_specification_key
            , bid_code
            , COUNT(*) AS row_count
            , ROW_NUMBER() OVER (
                PARTITION BY item_specification_key
                ORDER BY COUNT(*) DESC, bid_code
              ) AS bid_code_rank
        FROM keyed
        WHERE bid_code IS NOT NULL
        GROUP BY
              item_specification_key
            , bid_code
    )

    , canonical_bid_code AS (
        SELECT
              item_specification_key
            , bid_code
        FROM canonical_bid_code_ranked
        WHERE bid_code_rank = 1
    )

    , canonical_bid_item_description_ranked AS (
        SELECT
              item_specification_key
            , bid_item_description
            , COUNT(*) AS row_count
            , ROW_NUMBER() OVER (
                PARTITION BY item_specification_key
                ORDER BY COUNT(*) DESC, bid_item_description
              ) AS bid_item_desc_rank
        FROM keyed
        WHERE bid_item_description IS NOT NULL
        GROUP BY
              item_specification_key
            , bid_item_description
    )

    , canonical_bid_item_description AS (
        SELECT
              item_specification_key
            , bid_item_description
        FROM canonical_bid_item_description_ranked
        WHERE bid_item_desc_rank = 1
    )

    , canonical_spec_book_year_ranked AS (
        SELECT
              item_specification_key
            , spec_book_year
            , COUNT(*) AS row_count
            , ROW_NUMBER() OVER (
                PARTITION BY item_specification_key
                ORDER BY COUNT(*) DESC, spec_book_year DESC NULLS LAST
              ) AS spec_book_year_rank
        FROM keyed
        WHERE spec_book_year IS NOT NULL
        GROUP BY
              item_specification_key
            , spec_book_year
    )

    , canonical_spec_book_year AS (
        SELECT
              item_specification_key
            , spec_book_year
        FROM canonical_spec_book_year_ranked
        WHERE spec_book_year_rank = 1
    )

    , detail_rollup AS (
        SELECT
              item_specification_key
            , specification_code
            , specification_description
            , effective_specification_description
            , effective_specification_description_standardized
            , measurement_unit
            , measurement_unit_standardized

            , COUNT(*) AS source_row_count
            , COUNT(DISTINCT base_row_key) AS distinct_base_row_key_count
            , COUNT(DISTINCT source_row_id) AS distinct_source_row_id_count
            , COUNT(DISTINCT project_id) AS project_count
            , COUNT(DISTINCT CASE WHEN is_engineer_estimate_row = FALSE THEN project_id END) AS vendor_project_count
            , COUNT(DISTINCT CASE WHEN is_engineer_estimate_row = TRUE THEN project_id END) AS estimate_project_count
            , COUNT(DISTINCT CASE WHEN is_engineer_estimate_row = FALSE THEN vendor_name_standardized END) AS vendor_count
            , COUNT(DISTINCT bid_code) AS distinct_bid_code_count
            , COUNT(DISTINCT bid_item_description) AS distinct_bid_item_description_count
            , COUNT(DISTINCT spec_book_year) AS distinct_spec_book_year_count
            , MAX(source_updated_at) AS latest_source_updated_at
            , MAX(ingested_at_utc) AS latest_ingested_at_utc
        FROM keyed
        GROUP BY
              item_specification_key
            , specification_code
            , specification_description
            , effective_specification_description
            , effective_specification_description_standardized
            , measurement_unit
            , measurement_unit_standardized
    )

    , vendor_latest_stats AS (
        SELECT
              item_specification_key
            , COUNT(*) AS vendor_latest_item_row_count
            , COUNT(DISTINCT project_id) AS vendor_latest_project_count
            , COUNT(DISTINCT vendor_name_standardized) AS vendor_latest_vendor_count
        FROM {VENDOR_LATEST_TABLE}
        WHERE item_specification_key IS NOT NULL
        GROUP BY item_specification_key
    )

    , estimate_current_stats AS (
        SELECT
              item_specification_key
            , COUNT(*) AS estimate_current_item_row_count
            , COUNT(DISTINCT project_id) AS estimate_current_project_count
        FROM {ESTIMATE_CURRENT_TABLE}
        WHERE item_specification_key IS NOT NULL
        GROUP BY item_specification_key
    )

    SELECT
          d.item_specification_key
        , d.specification_code
        , d.specification_description
        , d.effective_specification_description
        , d.effective_specification_description_standardized
        , d.measurement_unit
        , d.measurement_unit_standardized

        , cbc.bid_code AS default_bid_code
        , cbid.bid_item_description AS default_bid_item_description
        , csby.spec_book_year AS default_spec_book_year

        , d.project_count
        , d.vendor_project_count
        , d.estimate_project_count
        , d.vendor_count

        , COALESCE(vs.vendor_latest_item_row_count, 0) AS vendor_latest_item_row_count
        , COALESCE(vs.vendor_latest_project_count, 0) AS vendor_latest_project_count
        , COALESCE(vs.vendor_latest_vendor_count, 0) AS vendor_latest_vendor_count

        , COALESCE(es.estimate_current_item_row_count, 0) AS estimate_current_item_row_count
        , COALESCE(es.estimate_current_project_count, 0) AS estimate_current_project_count

        , d.distinct_bid_code_count
        , d.distinct_bid_item_description_count
        , d.distinct_spec_book_year_count

        , CASE WHEN d.distinct_bid_code_count > 1 THEN TRUE ELSE FALSE END AS has_multiple_bid_codes
        , CASE WHEN d.distinct_bid_item_description_count > 1 THEN TRUE ELSE FALSE END AS has_multiple_bid_item_descriptions
        , CASE WHEN d.distinct_spec_book_year_count > 1 THEN TRUE ELSE FALSE END AS has_multiple_spec_book_years

        , d.source_row_count
        , d.distinct_base_row_key_count
        , d.distinct_source_row_id_count
        , d.latest_source_updated_at
        , d.latest_ingested_at_utc

    FROM detail_rollup d
    LEFT JOIN canonical_bid_code cbc
        ON d.item_specification_key = cbc.item_specification_key
    LEFT JOIN canonical_bid_item_description cbid
        ON d.item_specification_key = cbid.item_specification_key
    LEFT JOIN canonical_spec_book_year csby
        ON d.item_specification_key = csby.item_specification_key
    LEFT JOIN vendor_latest_stats vs
        ON d.item_specification_key = vs.item_specification_key
    LEFT JOIN estimate_current_stats es
        ON d.item_specification_key = es.item_specification_key
    """)

    spark.sql(f"""
    COMMENT ON TABLE {TARGET_TABLE} IS
    'Item specification dimension built from silver.closed_bids_base_clean and enriched with vendor latest and estimate current usage statistics.'
    """)

    row_count = spark.sql(f"SELECT COUNT(*) AS row_count FROM {TARGET_TABLE}").collect()[0]["row_count"]

    log_run(
        "SUCCESS",
        row_count,
        f"Built {TARGET_TABLE} successfully."
    )

    print("=" * 70)
    print(f"Built {TARGET_TABLE}")
    print(f"Row count: {row_count:,}")
    print("=" * 70)

except Exception as exc:
    log_run("FAILED", None, str(exc))
    raise